# without spliting attack

# Mount Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Imports

In [ ]:
#!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 922.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 95.3 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [1]:
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt

# Paths

In [2]:
BASE_PATH = "../Data/"
DATA_PATH = os.path.join(BASE_PATH, "swat_preprocessed")
MODEL_PATH = os.path.join(BASE_PATH, "models")

os.makedirs(MODEL_PATH, exist_ok=True)

# Load Windowed Data

In [3]:
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))


In [4]:
X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))


In [5]:
X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))


In [6]:
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))



# Build Autoencoder

In [7]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_size=64):
        super().__init__()

        self.encoder = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, n_features, batch_first=True)

    def forward(self, x):
        encoded, _ = self.encoder(x)
        decoded, _ = self.decoder(encoded)
        return decoded

In [9]:
import torch

X_train_t = torch.from_numpy(X_train).float()
X_val_t   = torch.from_numpy(X_val).float()
X_test_t  = torch.from_numpy(X_test).float()

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import gc

# ====== CONFIG ======
BATCH_SIZE = 32
EPOCHS = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
# ====== INIT MODEL (FIX) ======
n_features = X_train.shape[2]

model = LSTMAutoencoder(n_features=n_features, hidden_size=64)
model = model.to(DEVICE)

# ====== LOSS + OPTIMIZER (FIX) ======
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ====== DATASET ======
train_dataset = TensorDataset(X_train_t, X_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

if 'X_val_t' in globals():
    val_dataset = TensorDataset(X_val_t, X_val_t)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
else:
    val_loader = None

# ====== TRAINING ======
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        output = model(xb)
        loss = criterion(output, yb)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        del xb, yb, output, loss

    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)

    # ====== VALIDATION ======
    if val_loader is not None:
        model.eval()
        val_epoch_loss = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                output = model(xb)
                loss = criterion(output, yb)

                val_epoch_loss += loss.item()

                del xb, yb, output, loss

        val_epoch_loss /= len(val_loader)
        val_losses.append(val_epoch_loss)

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {epoch_loss:.6f} | Val Loss: {val_epoch_loss:.6f}")
    else:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {epoch_loss:.6f}")

    # ====== CLEANUP ======
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



Using device: cuda
Epoch [1/20] Train Loss: 0.414218 | Val Loss: 0.544490
Epoch [2/20] Train Loss: 0.412168 | Val Loss: 0.544147
Epoch [3/20] Train Loss: 0.412072 | Val Loss: 0.543954
Epoch [4/20] Train Loss: 0.412028 | Val Loss: 0.543897
Epoch [5/20] Train Loss: 0.411993 | Val Loss: 0.543785
Epoch [6/20] Train Loss: 0.411965 | Val Loss: 0.543694
Epoch [7/20] Train Loss: 0.411944 | Val Loss: 0.543708
Epoch [8/20] Train Loss: 0.411925 | Val Loss: 0.543805
Epoch [9/20] Train Loss: 0.411911 | Val Loss: 0.543662
Epoch [10/20] Train Loss: 0.411907 | Val Loss: 0.543658
Epoch [11/20] Train Loss: 0.411899 | Val Loss: 0.543652
Epoch [12/20] Train Loss: 0.411894 | Val Loss: 0.543644
Epoch [13/20] Train Loss: 0.411883 | Val Loss: 0.543631
Epoch [14/20] Train Loss: 0.411880 | Val Loss: 0.543635
Epoch [15/20] Train Loss: 0.411880 | Val Loss: 0.543606
Epoch [16/20] Train Loss: 0.411897 | Val Loss: 0.543638
Epoch [17/20] Train Loss: 0.411885 | Val Loss: 0.543600
Epoch [18/20] Train Loss: 0.411879 | V

In [11]:
batch_size = 32  # try 16 or 8 if still OOM

model.eval()
results = []

with torch.no_grad():
    for i in range(0, len(X_test_t), batch_size):
        batch = X_test_t[i:i+batch_size].to(DEVICE)

        recon = model(batch)

        err = ((recon - batch) ** 2).mean(dim=(1,2))
        results.append(err.cpu())

errors = torch.cat(results).numpy()

In [12]:
batch_size = 32  # reduce if needed

model.eval()
val_results = []

with torch.no_grad():
    for i in range(0, len(X_val_t), batch_size):
        batch = X_val_t[i:i+batch_size].to(DEVICE)

        recon = model(batch)

        err = ((recon - batch) ** 2).mean(dim=(1,2))
        val_results.append(err.cpu())

val_errors = torch.cat(val_results).numpy()

In [13]:
threshold = np.percentile(val_errors, 95)

In [14]:
y_pred = (errors > threshold).astype(int)

In [15]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.87      0.93      0.90     69161
         1.0       0.90      0.81      0.85     51020

    accuracy                           0.88    120181
   macro avg       0.88      0.87      0.87    120181
weighted avg       0.88      0.88      0.88    120181



In [16]:
joblib.dump(model.state_dict(), os.path.join(MODEL_PATH, "lstm_autoencoder.pth"))

['../Data/models\\lstm_autoencoder.pth']